# Mô Hình Dự Báo Doanh Thu (XGBoost + Recursive Forecasting)
Notebook này thực hiện huấn luyện mô hình dự báo doanh thu và giá vốn (COGS) theo ngày, sử dụng thuật toán XGBoost và phương pháp dự báo đệ quy.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")



## 1. CONFIGURATION


In [ ]:
# --- 1. CONFIGURATION ---
DATA_DIR = "data"
SALES_PATH = os.path.join(DATA_DIR, "sales.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
OUTPUT_SUB_PATH = "submission.csv"
PLOT_DIR = "plots"

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)



## 2. DATA LOADING


In [ ]:
# --- 2. DATA LOADING ---
print("Loading data...")
sales_df = pd.read_csv(SALES_PATH)
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
sales_df = sales_df.sort_values('Date').reset_index(drop=True)

sub_df = pd.read_csv(SUBMISSION_PATH)
sub_df['Date'] = pd.to_datetime(sub_df['Date'])

# Create a combined timeline from the start of sales to the end of submission
all_dates = pd.concat([
    sales_df[['Date', 'Revenue', 'COGS']], 
    pd.DataFrame({
        'Date': sub_df['Date'], 
        'Revenue': np.nan, 
        'COGS': np.nan
    })
], ignore_index=True)



## 3. FEATURE ENGINEERING


In [ ]:
# --- 3. FEATURE ENGINEERING ---
print("Extracting Time-based features...")
def create_time_features(df):
    df = df.copy()
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['DayOfMonth'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month
    df['Quarter'] = df['Date'].dt.quarter
    df['Year'] = df['Date'].dt.year
    df['DayOfYear'] = df['Date'].dt.dayofyear
    
    # Flags
    df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
    df['Is_Month_Start'] = df['Date'].dt.is_month_start.astype(int)
    df['Is_Month_End'] = df['Date'].dt.is_month_end.astype(int)
    
    # Fourier terms for seasonality (yearly)
    df['sin_365'] = np.sin(2 * np.pi * df['DayOfYear'] / 365.25)
    df['cos_365'] = np.cos(2 * np.pi * df['DayOfYear'] / 365.25)
    
    # Trend
    base_date = pd.to_datetime('2012-07-04')
    df['Time_Index'] = (df['Date'] - base_date).dt.days
    
    # --- HOLIDAY FEATURES ---
    # 1. Các ngày lễ dương lịch cố định (Lễ tình nhân, 8/3, 30/4, 1/5, Quốc khánh, 20/10, Giáng sinh...)
    solar_holidays = [
        (1, 1), (2, 14), (3, 8), (4, 30), (5, 1), 
        (9, 2), (10, 20), (12, 24), (12, 25)
    ]
    df['Is_Solar_Holiday'] = df.apply(lambda row: 1 if (row['Month'], row['DayOfMonth']) in solar_holidays else 0, axis=1)
    
    # 2. Black Friday (Ngày thứ 6 lần thứ 4 của tháng 11) - Đỉnh điểm Sale Thương mại điện tử
    df['Is_Black_Friday'] = ((df['Month'] == 11) & 
                             (df['DayOfWeek'] == 4) & 
                             (df['DayOfMonth'] >= 22) & 
                             (df['DayOfMonth'] <= 28)).astype(int)
    
    # 3. Đếm ngược đến Tết Nguyên Đán (Lunar New Year)
    # Hardcode ngày Mùng 1 Tết các năm từ 2012 đến 2025
    tet_dates = pd.to_datetime([
        '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08',
        '2017-01-28', '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12',
        '2022-02-01', '2023-01-22', '2024-02-10', '2025-01-29'
    ])
    
    def get_days_to_tet(current_date):
        future_tets = tet_dates[tet_dates >= current_date]
        if len(future_tets) == 0: 
            return -1
        days_diff = (future_tets[0] - current_date).days
        # Ta chỉ quan tâm hiệu ứng mua sắm trong vòng 30 ngày trước Tết
        return days_diff if days_diff <= 30 else -1

    df['Days_To_Tet'] = df['Date'].apply(get_days_to_tet)
    
    return df

def create_lag_features(df, targets):
    df = df.copy()
    for col in targets:
        df[f'{col}_Lag_1'] = df[col].shift(1)
        df[f'{col}_Lag_7'] = df[col].shift(7)
        df[f'{col}_Lag_30'] = df[col].shift(30)
        df[f'{col}_Lag_364'] = df[col].shift(364)
        df[f'{col}_Rolling_Mean_7'] = df[col].shift(1).rolling(window=7).mean()
        df[f'{col}_Rolling_Mean_30'] = df[col].shift(1).rolling(window=30).mean()
    return df

targets = ['Revenue', 'COGS']

# We only process lags for historical data initially
hist_df = all_dates[all_dates['Date'] < '2023-01-01'].copy()
hist_df = create_time_features(hist_df)
hist_df = create_lag_features(hist_df, targets)

# Drop rows with NaNs caused by the 365-day lag
train_df = hist_df.dropna().reset_index(drop=True)

# Train/Val Split (Use 2022 as Validation)
val_start = '2022-01-01'
train_data = train_df[train_df['Date'] < val_start]
val_data = train_df[train_df['Date'] >= val_start]

features = [c for c in train_df.columns if c not in ['Date', 'Revenue', 'COGS']]



## 4. MODEL TRAINING WITH HYPERPARAMETER TUNING


In [ ]:
# --- 4. MODEL TRAINING WITH HYPERPARAMETER TUNING ---
import random
import itertools
from sklearn.model_selection import TimeSeriesSplit

models = {}

# Define Hyperparameter Grid for Random Search
param_grid = {
    'learning_rate': [0.005, 0.01, 0.03, 0.05, 0.1],
    'max_depth': [4, 5, 6, 7, 8, 10],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7, 10],
    'gamma': [0, 0.1, 0.2, 0.3, 0.5]
}
# Thiết lập n_iter_search cao hơn (ví dụ 50 hoặc 100) để tận dụng Colab và train vài tiếng
n_iter_search = 300
n_splits = 4 # 4-fold time-series cross-validation

for target in targets:
    print(f"\n{'='*50}")
    print(f"--- Hyperparameter Tuning & Training for {target} ---")
    print(f"{'='*50}")
    
    # Dùng toàn bộ dữ liệu train_data cho CV
    X_cv_full = train_data[features].reset_index(drop=True)
    y_cv_full = np.log1p(train_data[target]).reset_index(drop=True)
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    best_score = float('inf')
    best_params = None
    best_n_estimators = 1500
    
    # Tạo ngẫu nhiên các tập hợp hyperparameters
    random.seed(42)
    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    sampled_params = random.sample(param_combinations, min(n_iter_search, len(param_combinations)))
    
    print(f"Testing {len(sampled_params)} hyperparameter combinations using {n_splits}-fold TimeSeriesSplit...")
    print("Quá trình này có thể mất vài tiếng trên Colab. Vui lòng chờ...\n")
    
    for idx, params in enumerate(sampled_params):
        cv_scores = []
        best_iters = []
        
        for train_idx, val_idx in tscv.split(X_cv_full):
            X_cv_train, y_cv_train = X_cv_full.iloc[train_idx], y_cv_full.iloc[train_idx]
            X_cv_val, y_cv_val = X_cv_full.iloc[val_idx], y_cv_full.iloc[val_idx]
            
            model = xgb.XGBRegressor(
                n_estimators=3000, 
                early_stopping_rounds=50,
                random_state=42,
                n_jobs=-1,
                **params
            )
            
            model.fit(
                X_cv_train, y_cv_train,
                eval_set=[(X_cv_val, y_cv_val)],
                verbose=False
            )
            
            preds = model.predict(X_cv_val)
            rmse = np.sqrt(mean_squared_error(y_cv_val, preds))
            cv_scores.append(rmse)
            best_iters.append(getattr(model, 'best_iteration', 3000))
            
        avg_rmse = np.mean(cv_scores)
        avg_iter = int(np.mean(best_iters))
        
        if avg_rmse < best_score:
            best_score = avg_rmse
            best_params = params
            best_n_estimators = avg_iter
            
        if (idx + 1) % 10 == 0 or (idx + 1) == len(sampled_params):
            print(f"  [Iter {idx+1:03d}/{len(sampled_params)}] Best CV RMSE: {best_score:.4f} | Optimal Trees: {best_n_estimators} | Best Params: {best_params}")
            
    print(f"\n[DONE] Best parameters for {target}:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"Optimal n_estimators: {best_n_estimators}")
    print(f"Best CV RMSE (log scale): {best_score:.4f}")
    
    # ---------------------------------------------------------
    # Huấn luyện mô hình cuối cùng với parameters tốt nhất
    # Dùng val_data (2022) để early stopping và đánh giá
    # ---------------------------------------------------------
    print(f"\nTraining Final Model for {target} on Train/Val split with best parameters...")
    
    X_train_final = train_data[features]
    y_train_final = np.log1p(train_data[target])
    
    X_val_final = val_data[features]
    y_val_final = np.log1p(val_data[target])
    
    final_model = xgb.XGBRegressor(
        # Cộng thêm một chút dự phòng cho early stopping
        n_estimators=best_n_estimators + 200, 
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1,
        **best_params
    )
    
    final_model.fit(
        X_train_final, y_train_final,
        eval_set=[(X_val_final, y_val_final)],
        verbose=100
    )
    
    # Validation Evaluation
    preds_log = final_model.predict(X_val_final)
    preds = np.expm1(preds_log)
    y_val_exp = np.expm1(y_val_final)
    
    mae = mean_absolute_error(y_val_exp, preds)
    rmse = np.sqrt(mean_squared_error(y_val_exp, preds))
    r2 = r2_score(y_val_exp, preds)
    print(f"\nFinal Validation {target} - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.4f}")
    
    models[target] = final_model
    
    # Feature Importance Plot
    plt.figure(figsize=(10, 8))
    xgb.plot_importance(final_model, max_num_features=15, importance_type='weight')
    plt.title(f'Feature Importance ({target})')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'feature_importance_{target}.png'))
    plt.close()
    
    # SHAP Explanations
    print(f"Generating SHAP plots for {target}...")
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_val_final)
    
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_val_final, show=False)
    plt.title(f'SHAP Summary ({target})')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'shap_summary_{target}.png'))
    plt.close()



## 5. RECURSIVE FORECASTING


In [ ]:
# --- 5. RECURSIVE FORECASTING ---
print("\nStarting Recursive Forecasting for 2023-2024...")
test_dates = sub_df['Date'].sort_values().reset_index(drop=True)

# Maintain working_df for recursive updates
working_df = hist_df.copy()

predictions = { 'Date': [], 'Revenue': [], 'COGS': [] }

for current_date in test_dates:
    # Append the new row with Date only
    new_row_df = pd.DataFrame([{'Date': current_date, 'Revenue': np.nan, 'COGS': np.nan}])
    new_row_df = create_time_features(new_row_df)
    
    # Find index where the new row will be appended
    idx = len(working_df)
    working_df = pd.concat([working_df, new_row_df], ignore_index=True)
    
    # Calculate Lag Features recursively for the new row
    for target in targets:
        working_df.loc[idx, f'{target}_Lag_1'] = working_df.loc[idx-1, target]
        working_df.loc[idx, f'{target}_Lag_7'] = working_df.loc[idx-7, target]
        working_df.loc[idx, f'{target}_Lag_30'] = working_df.loc[idx-30, target]
        working_df.loc[idx, f'{target}_Lag_364'] = working_df.loc[idx-364, target]
        
        # Rolling means
        working_df.loc[idx, f'{target}_Rolling_Mean_7'] = working_df.loc[idx-7:idx-1, target].mean()
        working_df.loc[idx, f'{target}_Rolling_Mean_30'] = working_df.loc[idx-30:idx-1, target].mean()
        
    X_test = working_df.loc[[idx], features]
    
    for target in targets:
        pred_log = models[target].predict(X_test)[0]
        working_df.loc[idx, target] = np.expm1(pred_log)
        
    predictions['Date'].append(current_date)
    predictions['Revenue'].append(working_df.loc[idx, 'Revenue'])
    predictions['COGS'].append(working_df.loc[idx, 'COGS'])



## 6. SUBMISSION FORMATTING


In [ ]:
# --- 6. SUBMISSION FORMATTING ---
print("\nSaving submission file...")
pred_df = pd.DataFrame(predictions)

# Merge back with the original submission dataframe to ensure order and format
final_sub = pd.merge(sub_df[['Date']], pred_df, on='Date', how='left')

# Format Date back to string if necessary, but to_csv handles datetime
final_sub.to_csv(OUTPUT_SUB_PATH, index=False)
print(f"Done! Submission saved to {OUTPUT_SUB_PATH}")
print(f"Plots saved to {PLOT_DIR}/ directory.")
